# TGT-M: the implementation, shown live (Review 2 demo, about 5 minutes)
**Team 7 · 21CSC314P Big Data Essentials.** Title: *TGT-M: A Temporal Gated Transformer Model for Seasonal Groundwater Stress Assessment and Cross-Region Generalization Across India Using Multimodal Data*.

This notebook **shows the real code and recomputes the published numbers**; it does not retrain the whole study (that takes hours and is not needed for the review).

| Section | What it shows |
|---|---|
| 1 | Finds the project files (zip upload, Kaggle dataset, or GitHub) |
| 2 | The data, and the **anomaly / stress label computed by hand** and matched with the pipeline (baseline uses only earlier readings) |
| 3 | What the model receives: a **13-step sequence tensor** whose current reading is hidden; no location columns |
| 4 | The **source code** of the TGT-M building blocks |
| 5 | **Build TGT-M**, count parameters, run one forward pass and see every shape |
| 6 | A **mini training run** (6,000 rows, about a minute) to show the training loop learns |
| 7 | The **report's test errors recomputed** from saved predictions and checked against the stored tables |
| 8 | The two **pre-registered tests** (tables) and the verdicts |


## 1 · Find the project files
Colab: upload `bde_bundle_core.zip` in the Files panel. Kaggle: add it as a dataset (Add Data). The cell finds it automatically.

In [ ]:
import os, sys, glob, zipfile, subprocess, time, shutil
from pathlib import Path
BUNDLE_ZIP = None      # e.g. "/content/bde_bundle_core.zip" if auto-detect fails
GITHUB_URL = "https://github.com/dhruvvvgg/bde-team7.git"   # only works after the project was pushed

ON_KAGGLE = Path("/kaggle").exists()
ON_COLAB = (not ON_KAGGLE) and Path("/content").exists()
WORK = Path("/kaggle/working") if ON_KAGGLE else Path("/content") if ON_COLAB else Path.cwd()

def looks_ok(p):
    p = Path(p); return (p / "modeling" / "data.py").exists() and (p / "processed" / "gw_features_by_state").exists()

def writable(c):
    """Kaggle mounts datasets read-only; copy the project to the writable working folder so quick/full mode can write results."""
    c = Path(c)
    if str(c).startswith("/kaggle/input"):
        dest = WORK / "bde-team7"
        if not looks_ok(dest):
            print("copying the project from the read-only dataset to", dest, "...")
            shutil.copytree(c, dest, dirs_exist_ok=True)
        return dest
    return c

def find_root():
    cands = [os.environ.get("BDE_ROOT"), Path.cwd(), Path.cwd().parent, WORK / "bde-team7"]
    cands += [Path(p).parent.parent for p in glob.glob("/kaggle/input/**/modeling/data.py", recursive=True)]
    for c in cands:
        if c and looks_ok(c): return writable(c)
    zips = ([BUNDLE_ZIP] if BUNDLE_ZIP else []) + glob.glob("/kaggle/input/**/bde_bundle_core*.zip", recursive=True) + glob.glob("/content/**/bde_bundle_core*.zip", recursive=True) + glob.glob(str(Path.cwd() / "bde_bundle_core*.zip"))
    for z in zips:
        if z and Path(z).exists():
            print("unzipping", Path(z).name); zipfile.ZipFile(z).extractall(WORK)
            for c in (WORK / "bde-team7", WORK):
                if looks_ok(c): return c
    dest = WORK / "bde-team7"
    if not dest.exists(): subprocess.run(["git", "clone", "-q", "--depth", "1", GITHUB_URL, str(dest)])
    if looks_ok(dest): return dest
    raise SystemExit("Project files not found: upload bde_bundle_core.zip (Colab: Files panel; Kaggle: Add Data) and rerun.")

ROOT = find_root(); os.environ["BDE_ROOT"] = str(ROOT); os.chdir(ROOT)
print("platform:", "Kaggle" if ON_KAGGLE else "Colab" if ON_COLAB else "local machine", "| project folder:", ROOT.name, "| cpus:", os.cpu_count())

In [ ]:
import importlib
miss = [p for m, p in {"xgboost": "xgboost", "pyarrow": "pyarrow", "sklearn": "scikit-learn", "scipy": "scipy"}.items() if importlib.util.find_spec(m) is None]
if miss: subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + miss)
import torch                                   # import torch BEFORE pandas (matters on Windows; harmless elsewhere)
import numpy as np, pandas as pd
sys.path.insert(0, str(ROOT / "modeling")); sys.path.insert(0, str(ROOT / "src"))
print("torch", torch.__version__, "| pandas", pd.__version__, "| numpy", np.__version__)

## 2 · The data and the label, computed by hand
One real well: **Cherfully, Thane (Maharashtra)**, a dug well. We recompute the May-2006 baseline, anomaly (z) and stress label ourselves and compare with the stored pipeline values.
Rule: *baseline = mean and SD of the **earlier** May readings (at least 5)*; *z = (reading − mean) / SD*; *High Stress if z ≥ 2*.

In [ ]:
m = pd.read_parquet(ROOT / "processed" / "gw_features_by_state",
                    columns=["well_id", "station_name", "district", "season", "date", "period_label", "gwl_m_bgl", "baseline_mean_m", "baseline_std_m", "anomaly_z", "stress_category"])
w = m[(m.well_id == "Wa62f7df166") & (m.season == "pre_monsoon")].sort_values("date").reset_index(drop=True)
print(w.station_name.iloc[0], "|", w.district.iloc[0])
i = w.index[w.period_label == "May-06"][0]
earlier = w.loc[:i - 1, "gwl_m_bgl"].dropna()
mean, sd = earlier.mean(), earlier.std(ddof=1)          # sample SD, only EARLIER readings
z = (w.gwl_m_bgl[i] - mean) / max(sd, 0.10)
print("earlier May readings used:", earlier.round(2).tolist())
print(f"computed here : mean {mean:.3f}  SD {sd:.3f}  z {z:.3f}  -> {'High Stress' if z >= 2 else 'not High Stress'}")
print(f"pipeline value: mean {w.baseline_mean_m[i]:.3f}  SD {w.baseline_std_m[i]:.3f}  z {w.anomaly_z[i]:.3f}  -> {w.stress_category[i]}")
display(w[["period_label", "gwl_m_bgl", "baseline_mean_m", "baseline_std_m", "anomaly_z", "stress_category"]].round(2).head(14))

In [ ]:
print("rows:", len(m), "| wells:", m.well_id.nunique(), "| readings:", int(m.gwl_m_bgl.notna().sum()), "| missing: %.2f%%" % (100 * m.gwl_m_bgl.isna().mean()))
print(m.stress_category.astype(str).value_counts().to_string())

## 3 · What the model receives
`modeling/data.py` builds, for every eligible row, a **tabular vector** and a **sequence tensor**: 13 steps (12 past rounds + the current round) × 10 channels. The current round's **reading is always hidden** (mask 0); only its season and rainfall up to the previous month are visible. Depth lags are expressed relative to the well's own baseline, and latitude, longitude, state and district are **never** inputs.

In [ ]:
import data as D
b = D.build(); meta = b["meta"]
print("eligible rows:", len(meta), "| tabular inputs A / B:", len(b["names"]) - int(b["is_rain"].sum()), "/", len(b["names"]), "| sequence tensor:", b["seq"].shape)
print("splits:", meta.split_chrono.value_counts().to_dict())
banned = {"lat", "lon", "state", "district", "year"}
print("forbidden location/year columns among the inputs:", sorted(banned & set(b["names"])) or "none")
k = int(np.flatnonzero((meta.well_id == "Wa62f7df166") & (meta.period_label == "May-06"))[0])
ch = ["reading(rel)", "mask", "Jan", "May", "Aug", "Nov", "valid", "log rain1m", "log rain3m", "spare"]
print("\nsample: well Wa62f7df166, round May-06 (last row = the round to predict, reading hidden)")
display(pd.DataFrame(b["seq"][k], columns=ch, index=[f"t-{12 - j}" if j < 12 else "t (hidden)" for j in range(13)]).round(2))

## 4 · The TGT-M source code
The building blocks as implemented in `modeling/nets.py` (GRU-D encoder, variable-selection network, gated residual network, and the full model).

In [ ]:
import inspect, nets as N
for obj in (N.GRUD, N.GRN, N.VSN2):
    print(inspect.getsource(obj)); print("-" * 90)

In [ ]:
print(inspect.getsource(N.TGTM2))

## 5 · Build TGT-M and run one forward pass
`TGTM2` (exposed as `"TGT-M"`): **GRU-D** encoder (mask + time decay) → **variable-selection network** → **gated residual** enrichment → **attention** with the current round as query → heads: point, q10, q50, q90.

In [ ]:
import lib as Lb
cfg = Lb.GRIDS["TGT-M"][2]                     # configuration chosen on validation for Model B
Bq = Lb.Bundle(b, True, (meta.split_chrono == "train").to_numpy())
net = N.NETS["TGT-M"](n_static=Bq.X.shape[1], rain_idx=np.flatnonzero(Bq.is_rain), **cfg["arch"])
print("config:", cfg)
print("parameters:", f"{sum(p.numel() for p in net.parameters()):,}")
net.eval()
seq, st = torch.from_numpy(Bq.seq[:8]), torch.from_numpy(Bq.X[:8])
with torch.no_grad():
    Hs = net.enc(seq); c, wts = net.vsn(st); point, q = net(seq, st, return_all=True)
print("input sequence       :", tuple(seq.shape), "   static vector:", tuple(st.shape))
print("GRU-D states         :", tuple(Hs.shape), "   (one state per step)")
print("VSN context / weights:", tuple(c.shape), "/", tuple(wts.shape), "(weights sum to", round(float(wts.sum(1).mean()), 3), ")")
print("attention weights    :", tuple(net.last_attn.shape), "(the current round attends over the 13 steps)")
print("point / quantiles    :", tuple(point.shape), "/", tuple(q.shape), " (untrained network, random output)")

## 6 · Mini training run (6,000 rows, about a minute)
The same training function as the study (`lib.fit_deep`: AdamW, Huber + pinball loss, early stopping), on a small random sample so it finishes quickly. The published numbers come from the full runs.

In [ ]:
import warnings; warnings.filterwarnings("ignore")
tr = (meta.split_chrono == "train").to_numpy(); va = (meta.split_chrono == "val").to_numpy()
rng = np.random.default_rng(0)
tr_s = np.zeros(len(meta), bool); tr_s[rng.choice(np.flatnonzero(tr), 6000, replace=False)] = True
va_s = np.zeros(len(meta), bool); va_s[rng.choice(np.flatnonzero(va), 2000, replace=False)] = True
y = meta["anomaly_z_capped"].to_numpy()
torch.manual_seed(0)
t0 = time.time()
net2, best, ep = Lb.fit_deep("TGT-M", Bq, y, tr_s, va_s, cfg, 0, epochs=6, patience=3, verbose=True)
print(f"trained on 6,000 rows in {time.time() - t0:.0f}s; best validation |z| MAE {best:.3f}  (always predicting z = 0 gives {np.abs(y[va_s]).mean():.3f})")

## 7 · The report's numbers, recomputed from saved predictions
`results/powerbi/test_predictions.csv` holds each model's predicted depth for every test row (seed 0). We recompute the depth MAE ourselves and compare it with the stored per-seed table (the same numbers as in the report).

In [ ]:
tp = pd.read_csv(ROOT / "results" / "powerbi" / "test_predictions.csv")
seed0 = pd.read_csv(ROOT / "results" / "tables" / "chrono_per_seed.csv"); seed0 = seed0[seed0.seed == 0].set_index(["model", "variant"])
rows = []
for col in [c for c in tp.columns if c.endswith("_depth_pred")]:
    mod, var = col.split("_")[0], col.split("_")[1]
    mine = float(np.abs(tp[col] - tp.gwl_m_bgl).mean())
    stored = float(seed0.loc[(mod, var), "depth_MAE"]) if (mod, var) in seed0.index else np.nan
    rows.append([mod, var, round(mine, 3), round(stored, 3), "match" if abs(mine - stored) < 1e-3 else "CHECK"])
out = pd.DataFrame(rows, columns=["model", "variant", "recomputed depth MAE (m)", "stored in report tables (m)", "check"]).sort_values(["variant", "recomputed depth MAE (m)"])
print("test rows:", len(tp), "| seasonal-baseline depth MAE for reference:", round(float(seed0.loc[("Seasonal baseline", "B"), "depth_MAE"]), 3), "m")
display(out)

In [ ]:
s = pd.read_csv(ROOT / "results" / "tables" / "chrono_summary.csv")
print("Mean over 3 seeds (the numbers quoted in the report), Model B = with rainfall:")
display(s[s.variant == "B"][["model", "depth_MAE_mean", "depth_MAE_std", "anom_R2_mean", "AUC_stressed_mean"]].round(3).sort_values("depth_MAE_mean").reset_index(drop=True))
print("TGT-M is 5th of 7 learned models on depth MAE; rainfall (B vs A) improves every model.")

## 8 · The two pre-registered tests (from the saved result tables)
Hypotheses, metrics and pass criteria were written down in `docs/preregistration_review2.md` before these tests were run.

In [ ]:
T = ROOT / "results" / "tables"
print("H1: 80% intervals (lower interval score is better; nominal coverage 0.80)")
display(pd.read_csv(T / "interval_summary.csv").round(3).head(6))
print("H2: anomaly MAE when 0-60% of the recent readings are removed at test time (all models retrained, identical masks)")
display(pd.read_csv(T / "robust_summary.csv").pivot(index="model", columns="p", values="mean").round(4))
print("Verdicts: H1 supported; H2a (smaller loss than LSTM and 1D-CNN) supported; H2b (lowest error at 60% removal) NOT supported.")

**End of demo.** Honest summary for the panel: TGT-M ranks 5th of 7 learned models on accuracy; its demonstrated strengths are better 80% intervals and a smaller loss than LSTM/CNN when recent readings are missing (pre-registered tests); rainfall improves every model.